# 05 — KPIs: Indicadores Clave de Desempeño — Análisis COVID-19

Se calculan **10 KPIs** sobre el dataset Chicago Crimes 2017–2025 usando los tres frameworks de procesamiento distribuido. Cada KPI compara las tres eras COVID (PRE / DURANTE / POST) para cuantificar el impacto de la pandemia en la criminalidad de Chicago. Al final se consolida la tabla `kpis_summary` en BigQuery.

| # | KPI | Framework | Categoría Power BI |
|---|-----|-----------|--------------------|
| 1 | Variación de la tasa de arresto global por era | Dask | Efectividad policial |
| 2 | Caída de criminalidad en el pico del confinamiento (2020) | Dask | Tendencia COVID |
| 3 | Aumento de violencia doméstica durante el COVID | Modin | Violencia doméstica |
| 4 | Tipo de crimen con mayor cambio de resolución entre eras | Modin | Efectividad policial |
| 5 | Redistribución geográfica: distritos que ganaron/perdieron crimen | Modin | Geografía |
| 6 | Recuperación post-COVID: años para volver al nivel PRE | Dask | Tendencia COVID |
| 7 | Cambio en el índice de criminalidad nocturna por era | Dask | Temporalidad |
| 8 | Variación de crímenes violentos (FBI Part I) por era | Spark | Efectividad policial |
| 9 | Diversidad de crímenes por era — ¿se concentraron durante el COVID? | Spark | Geografía |
| 10 | Variación en la calidad del registro CPD durante el COVID (MTTR) | Spark | Calidad de datos |

In [1]:
import dask.dataframe as dd
import modin.pandas as mpd
import pandas as pd
import numpy as np
import os
import ray
import warnings
warnings.filterwarnings('ignore')

PROJECT_ID     = 'my-first-project-492901'
DATASET_ID     = 'chicago_crimes_results'
FOLDER         = 'Chicago_Crimes_by_Year'
YEARS_ANALYSIS = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
ERA_MAP        = {2017:'PRE', 2018:'PRE', 2019:'PRE',
                  2020:'DURANTE', 2021:'DURANTE', 2022:'DURANTE',
                  2023:'POST', 2024:'POST', 2025:'POST'}

files = [os.path.join(FOLDER, f'Chicago_Crimes_{y}.csv') for y in YEARS_ANALYSIS]

def to_bigquery(df: pd.DataFrame, table_name: str, if_exists: str = 'replace') -> None:
    df.to_gbq(
        destination_table=f'{DATASET_ID}.{table_name}',
        project_id=PROJECT_ID,
        if_exists=if_exists,
        progress_bar=False,
    )
    print(f'  → BigQuery: {PROJECT_ID}.{DATASET_ID}.{table_name}  ({len(df):,} filas)')

kpis = []  # acumula los 10 KPIs para kpis_summary
print('Entorno listo. Período: 2017–2025 (PRE / DURANTE / POST COVID)')

2026-04-27 09:36:13,199	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Entorno listo. Período: 2017–2025 (PRE / DURANTE / POST COVID)


In [2]:
# ── Carga con Dask (KPIs 1, 2, 6, 7) ─────────────────────────────────────────
dtypes = {
    'unique_key': 'Int64', 'case_number': 'object', 'block': 'object',
    'iucr': 'object', 'primary_type': 'object', 'description': 'object',
    'location_description': 'object', 'beat': 'Int64', 'district': 'Int64',
    'ward': 'Int64', 'community_area': 'Int64', 'fbi_code': 'object',
    'x_coordinate': 'Int64', 'y_coordinate': 'Int64', 'year': 'Int64',
    'latitude': 'float64', 'longitude': 'float64', 'location': 'object',
}
ddf = dd.read_csv(files, dtype=dtypes, parse_dates=['date', 'updated_on'], assume_missing=True)
ddf['covid_era'] = ddf['year'].map(ERA_MAP, meta=('covid_era', 'object'))
total = len(ddf)
print(f'Total registros (Dask, 2017–2025): {total:,}')

Total registros (Dask, 2017–2025): 2,072,943


---
## KPI 1 — Variación de la Tasa de Arresto por Era COVID *(Dask)*

**Definición:** Tasa de arresto (crímenes con arrest=True / total) calculada independientemente para cada era. Se reporta la variación entre PRE y DURANTE como KPI central.

**Fórmula:** `Δ = arrest_rate_DURANTE - arrest_rate_PRE`

**Interpretación:** Una caída negativa en la era DURANTE indica que el COVID redujo la capacidad operativa del CPD (protocolos sanitarios, personal reducido, restricciones judiciales). La recuperación en POST mide la resiliencia institucional del departamento.

In [3]:
arrest_by_era = (
    ddf.groupby('covid_era')
       .agg({'arrest': 'sum', 'unique_key': 'count'})
       .compute()
       .rename(columns={'arrest': 'arrests', 'unique_key': 'total'})
       .reset_index()
)
arrest_by_era['arrest_rate_pct'] = (arrest_by_era['arrests'] / arrest_by_era['total'] * 100).round(2)

pre_rate     = float(arrest_by_era.loc[arrest_by_era['covid_era']=='PRE',    'arrest_rate_pct'].values[0])
durante_rate = float(arrest_by_era.loc[arrest_by_era['covid_era']=='DURANTE','arrest_rate_pct'].values[0])
post_rate    = float(arrest_by_era.loc[arrest_by_era['covid_era']=='POST',   'arrest_rate_pct'].values[0])
delta_kpi1   = round(durante_rate - pre_rate, 2)

print('KPI 1 | Variación de Tasa de Arresto por Era COVID')
for _, row in arrest_by_era.sort_values('covid_era').iterrows():
    print(f"  {row['covid_era']:<8}: {row['arrest_rate_pct']:.2f}%  ({int(row['total']):,} crímenes)")
print(f"  ΔDURANTE-PRE : {delta_kpi1:+.2f} pp")

kpis.append({'kpi_id': 1, 'kpi_name': 'Variación Tasa de Arresto PRE→DURANTE',
             'value_numeric': delta_kpi1,
             'value_text': f'PRE:{pre_rate:.2f}% → DURANTE:{durante_rate:.2f}% → POST:{post_rate:.2f}%',
             'unit': 'pp', 'framework': 'Dask'})

KPI 1 | Variación de Tasa de Arresto por Era COVID
  DURANTE : 13.42%  (661,583 crímenes)
  POST    : 13.32%  (611,521 crímenes)
  PRE     : 20.36%  (799,839 crímenes)
  ΔDURANTE-PRE : -6.94 pp


---
## KPI 2 — Caída de Criminalidad en el Pico del Confinamiento (2020) *(Dask)*

**Definición:** Variación porcentual interanual de crímenes en 2020 respecto a 2019 — el año de mayor impacto del confinamiento de primavera.

**Fórmula:** `(crímenes_2020 - crímenes_2019) / crímenes_2019 × 100`

**Interpretación:** El confinamiento de Illinois comenzó el 21 de marzo de 2020. Una caída superior al 10% en 2020 confirma el efecto directo del cierre sobre la actividad criminal exterior. Este es el KPI de referencia para validar la hipótesis central del análisis.

In [4]:
crimes_by_year = (
    ddf.groupby('year')['unique_key']
       .count().compute()
       .reset_index()
       .rename(columns={'unique_key': 'total'})
       .sort_values('year')
)
crimes_by_year['covid_era'] = crimes_by_year['year'].map(ERA_MAP)
crimes_by_year['yoy_pct']   = crimes_by_year['total'].pct_change() * 100

n2019 = int(crimes_by_year.loc[crimes_by_year['year']==2019, 'total'].values[0])
n2020 = int(crimes_by_year.loc[crimes_by_year['year']==2020, 'total'].values[0])
kpi2  = round((n2020 - n2019) / n2019 * 100, 2)

print('KPI 2 | Caída de Criminalidad en el Pico del Confinamiento (2020)')
print(f'  Crímenes 2019 : {n2019:,}')
print(f'  Crímenes 2020 : {n2020:,}')
print(f'  Variación     : {kpi2:+.2f}%')
print()
print('Evolución año a año:')
print(crimes_by_year[['year','covid_era','total','yoy_pct']].to_string(index=False))

kpis.append({'kpi_id': 2, 'kpi_name': 'Caída de Criminalidad en Confinamiento 2020',
             'value_numeric': kpi2, 'value_text': f'{kpi2:+.2f}% (2019→2020)',
             'unit': '%', 'framework': 'Dask'})

KPI 2 | Caída de Criminalidad en el Pico del Confinamiento (2020)
  Crímenes 2019 : 261,555
  Crímenes 2020 : 212,522
  Variación     : -18.75%

Evolución año a año:
 year covid_era  total    yoy_pct
 2017       PRE 269214       <NA>
 2018       PRE 269070  -0.053489
 2019       PRE 261555  -2.792954
 2020   DURANTE 212522 -18.746726
 2021   DURANTE 209406  -1.466201
 2022   DURANTE 239655  14.445145
 2023      POST 262756   9.639273
 2024      POST 256305  -2.455129
 2025      POST  92460 -63.925792


---
## KPI 3 — Aumento de Violencia Doméstica Durante el COVID *(Modin)*

**Definición:** Variación en puntos porcentuales de la tasa de violencia doméstica entre la era PRE y DURANTE.

**Fórmula:** `Δ = (domestic_DURANTE / total_DURANTE) - (domestic_PRE / total_PRE)` × 100

**Interpretación:** Este KPI es el más crítico socialmente del análisis. El confinamiento de 2020 fue predicho por organizaciones de derechos humanos como un catalizador de violencia doméstica. Un incremento confirmaría la necesidad de políticas de intervención social (teléfonos de crisis, refugios, órdenes de alejamiento remotas) durante cualquier futura emergencia de confinamiento.

In [5]:
ray.init(ignore_reinit_error=True)
mdf = mpd.concat([mpd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
mdf['covid_era']    = mdf['year'].map(ERA_MAP)
mdf['domestic_bool'] = mdf['domestic'].astype(str).str.lower().eq('true')

dom_by_era = (
    mdf.groupby('covid_era')
       .agg(domestic_crimes=('domestic_bool', 'sum'), total=('unique_key', 'count'))
       .reset_index()
)
dom_by_era['domestic_rate_pct'] = (dom_by_era['domestic_crimes'] / dom_by_era['total'] * 100).round(2)

pre_dom     = float(dom_by_era.loc[dom_by_era['covid_era']=='PRE',    'domestic_rate_pct'].values[0])
durante_dom = float(dom_by_era.loc[dom_by_era['covid_era']=='DURANTE','domestic_rate_pct'].values[0])
post_dom    = float(dom_by_era.loc[dom_by_era['covid_era']=='POST',   'domestic_rate_pct'].values[0])
delta_kpi3  = round(durante_dom - pre_dom, 2)

print('KPI 3 | Aumento de Violencia Doméstica Durante el COVID')
for _, row in dom_by_era.sort_values('covid_era').iterrows():
    print(f"  {row['covid_era']:<8}: {row['domestic_rate_pct']:.2f}%  ({int(row['domestic_crimes']):,} de {int(row['total']):,})")
print(f"  ΔDURANTE-PRE : {delta_kpi3:+.2f} pp")

kpis.append({'kpi_id': 3, 'kpi_name': 'Aumento de Violencia Doméstica DURANTE COVID',
             'value_numeric': delta_kpi3,
             'value_text': f'PRE:{pre_dom:.2f}% → DURANTE:{durante_dom:.2f}% → POST:{post_dom:.2f}%',
             'unit': 'pp', 'framework': 'Modin'})

2026-04-27 09:37:22,078	INFO worker.py:2012 -- Started a local Ray instance.


2026-04-27 09:37:23,298	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


KPI 3 | Aumento de Violencia Doméstica Durante el COVID
  DURANTE : 21.02%  (139,095 de 661,583)
  POST    : 18.39%  (112,471 de 611,521)
  PRE     : 19.20%  (153,534 de 799,839)
  ΔDURANTE-PRE : +1.82 pp


---
## KPI 4 — Tipo de Crimen con Mayor Cambio de Resolución entre Eras *(Modin)*

**Definición:** La categoría de crimen (`primary_type`) que experimentó el mayor cambio (positivo o negativo) en tasa de arresto entre PRE y DURANTE.

**Fórmula:** `MAX(|arrest_rate_DURANTE - arrest_rate_PRE|)` por primary_type

**Interpretación:** Identifica qué tipos de crimen fueron más afectados operativamente por el COVID. Un tipo con caída de >10pp en resolución durante el COVID indica que esa categoría fue especialmente difícil de atender bajo restricciones pandémicas. Un tipo con incremento podría indicar represalización de ciertos delitos domésticos al aumentar su detección.

In [6]:
mdf['arrest_bool'] = mdf['arrest'].astype(str).str.lower().eq('true')

rate_by_type_era = (
    mdf.groupby(['primary_type', 'covid_era'])
       .agg(arrests=('arrest_bool', 'sum'), total=('unique_key', 'count'))
       .reset_index()
)
rate_by_type_era['rate'] = (rate_by_type_era['arrests'] / rate_by_type_era['total'] * 100).round(2)

pivot_rate = rate_by_type_era.pivot(index='primary_type', columns='covid_era', values='rate').dropna()
pivot_rate['delta_durante_pre'] = (pivot_rate['DURANTE'] - pivot_rate['PRE']).round(2)
pivot_rate['abs_delta']         = pivot_rate['delta_durante_pre'].abs()
pivot_rate = pivot_rate.sort_values('abs_delta', ascending=False)

top_change = pivot_rate.iloc[0]
kpi4_type  = pivot_rate.index[0]
kpi4_delta = float(top_change['delta_durante_pre'])

print('KPI 4 | Tipo con Mayor Cambio de Resolución PRE→DURANTE')
print(f'  Tipo         : {kpi4_type}')
print(f'  PRE          : {top_change["PRE"]:.2f}%')
print(f'  DURANTE      : {top_change["DURANTE"]:.2f}%')
print(f'  Δ            : {kpi4_delta:+.2f} pp')
print('\nTop 10 tipos por magnitud de cambio:')
print(pivot_rate.head(10)[['PRE','DURANTE','POST','delta_durante_pre']].to_string())

kpis.append({'kpi_id': 4, 'kpi_name': 'Tipo con Mayor Cambio de Resolución por COVID',
             'value_numeric': kpi4_delta,
             'value_text': f'{kpi4_type}: {kpi4_delta:+.2f} pp (PRE→DURANTE)',
             'unit': 'pp', 'framework': 'Modin'})

KPI 4 | Tipo con Mayor Cambio de Resolución PRE→DURANTE
  Tipo         : PUBLIC PEACE VIOLATION
  PRE          : 66.49%
  DURANTE      : 38.61%
  Δ            : -27.88 pp

Top 10 tipos por magnitud de cambio:
                            PRE  DURANTE   POST  delta_durante_pre
primary_type                                                      
PUBLIC PEACE VIOLATION    66.49    38.61  46.96             -27.88
NON-CRIMINAL               5.19    30.00  44.44              24.81
CRIMINAL TRESPASS         56.53    33.48  29.06             -23.05
OTHER OFFENSE             21.55    13.00  18.36              -8.55
WEAPONS VIOLATION         71.03    62.63  61.42              -8.40
SEX OFFENSE               16.87     8.74   6.84              -8.13
ASSAULT                   17.52    10.38  10.56              -7.14
OTHER NARCOTIC VIOLATION  60.00    53.33  50.00              -6.67
OBSCENITY                 77.16    70.51  50.00              -6.65
STALKING                  12.90     6.34   5.31       

---
## KPI 5 — Redistribución Geográfica: Distrito que Más Cayó en DURANTE *(Modin)*

**Definición:** El distrito policial que experimentó la mayor caída porcentual de crímenes entre PRE y DURANTE — evidencia de la redistribución espacial del crimen durante el confinamiento.

**Fórmula:** `MIN( (crímenes_dist_DURANTE / años_DURANTE - crímenes_dist_PRE / años_PRE) / (crímenes_dist_PRE / años_PRE) × 100 )`

**Interpretación:** El distrito con mayor caída corresponde típicamente a zonas comerciales y turísticas que quedaron vacías durante el confinamiento. Este KPI confirma que el COVID no solo redujo el crimen total sino que lo redistribuyó geográficamente — información clave para ajustar los patrullajes por era.

In [7]:
dist_by_era = (
    mdf.dropna(subset=['district'])
       .groupby(['district', 'covid_era'])
       .size().reset_index(name='count')
)

# Normalizar por cantidad de años por era (3 años cada una)
dist_pivot = dist_by_era.pivot(index='district', columns='covid_era', values='count').dropna()
dist_pivot['annual_pre']     = dist_pivot['PRE']     / 3
dist_pivot['annual_durante'] = dist_pivot['DURANTE'] / 3
dist_pivot['delta_pct']      = ((dist_pivot['annual_durante'] - dist_pivot['annual_pre']) /
                                 dist_pivot['annual_pre'] * 100).round(2)
dist_pivot = dist_pivot.sort_values('delta_pct')

top_fall_dist = int(dist_pivot.index[0])
kpi5_delta    = float(dist_pivot.iloc[0]['delta_pct'])

print('KPI 5 | Redistribución Geográfica por COVID')
print(f'  Distrito con mayor caída   : Distrito {top_fall_dist}  ({kpi5_delta:+.2f}%)')
print(f'  Distrito con menor caída   : Distrito {int(dist_pivot.index[-1])}  ({dist_pivot.iloc[-1]["delta_pct"]:+.2f}%)')
print('\nTodos los distritos — variación anual promedio DURANTE vs PRE:')
print(dist_pivot[['annual_pre','annual_durante','delta_pct']].to_string())

kpis.append({'kpi_id': 5, 'kpi_name': 'Distrito con Mayor Caída de Crimen en COVID',
             'value_numeric': kpi5_delta,
             'value_text': f'Distrito {top_fall_dist}: {kpi5_delta:+.2f}% (DURANTE vs PRE anualizado)',
             'unit': '%', 'framework': 'Modin'})

KPI 5 | Redistribución Geográfica por COVID
  Distrito con mayor caída   : Distrito 1  (-33.89%)
  Distrito con menor caída   : Distrito 31  (+66.67%)

Todos los distritos — variación anual promedio DURANTE vs PRE:
            annual_pre  annual_durante  delta_pct
district                                         
1.0       15485.666667    10237.333333     -33.89
18.0      15514.000000    10509.000000     -32.26
14.0       9597.333333     7241.666667     -24.55
11.0      18699.000000    14212.666667     -23.99
10.0      12577.000000     9774.666667     -22.28
7.0       13979.333333    10903.333333     -22.00
15.0      10122.333333     8429.000000     -16.73
8.0       16269.666667    13649.333333     -16.11
25.0      13570.666667    11403.333333     -15.97
17.0       7455.333333     6272.000000     -15.87
5.0       11735.000000     9912.000000     -15.53
6.0       16687.000000    14154.000000     -15.18
9.0       11310.333333     9615.000000     -14.99
19.0      12086.000000    10483.000

---
## KPI 6 — Recuperación Post-COVID: Crímenes en 2023–2025 vs Nivel PRE *(Dask)*

**Definición:** Comparación del promedio anual de crímenes en la era POST (2023–2025) contra el promedio de la era PRE (2017–2019). Mide si Chicago recuperó los niveles pre-pandémicos.

**Fórmula:** `(avg_anual_POST - avg_anual_PRE) / avg_anual_PRE × 100`

**Interpretación:** Un valor cercano a 0% indica plena recuperación. Un valor negativo significa que el POST tiene menos crimen que el PRE — posible efecto de largo plazo de los cambios en movilidad y trabajo remoto. Un valor positivo significaría que el COVID dejó un "rebote" de criminalidad en la era posterior.

In [8]:
# crimes_by_year ya calculado en KPI 2
avg_pre  = float(crimes_by_year[crimes_by_year['covid_era']=='PRE']['total'].mean())
avg_post = float(crimes_by_year[crimes_by_year['covid_era']=='POST']['total'].mean())
kpi6     = round((avg_post - avg_pre) / avg_pre * 100, 2)

print('KPI 6 | Recuperación Post-COVID vs Nivel PRE')
print(f'  Promedio anual PRE  (2017–2019): {avg_pre:,.0f}')
print(f'  Promedio anual POST (2023–2025): {avg_post:,.0f}')
print(f'  Recuperación        : {kpi6:+.2f}%')
print()
status = 'POR ENCIMA del nivel PRE' if kpi6 > 5 else ('SIMILAR al nivel PRE' if abs(kpi6) <= 5 else 'POR DEBAJO del nivel PRE')
print(f'  Estado: {status}')

kpis.append({'kpi_id': 6, 'kpi_name': 'Recuperación Post-COVID vs Nivel PRE',
             'value_numeric': kpi6,
             'value_text': f'{kpi6:+.2f}% (POST vs PRE anualizado)',
             'unit': '%', 'framework': 'Dask'})

KPI 6 | Recuperación Post-COVID vs Nivel PRE
  Promedio anual PRE  (2017–2019): 266,613
  Promedio anual POST (2023–2025): 203,840
  Recuperación        : -23.54%

  Estado: POR DEBAJO del nivel PRE


---
## KPI 7 — Cambio en el Índice de Criminalidad Nocturna por Era *(Dask)*

**Definición:** Variación en el porcentaje de crímenes nocturnos (22:00–05:59) entre PRE y DURANTE COVID.

**Fórmula:** `Δ = (crímenes_nocturnos_DURANTE / total_DURANTE) - (crímenes_nocturnos_PRE / total_PRE)` × 100

**Interpretación:** El toque de queda nocturno en Illinois (vigente desde abril 2020 en algunos contextos) y el cierre de bares y restaurantes deberían haber reducido el crimen nocturno significativamente en la era DURANTE. Una caída de más de 2pp en la tasa nocturna confirma el efecto regulatorio del confinamiento sobre los patrones temporales del crimen.

In [9]:
ddf['hour']      = ddf['date'].dt.hour
ddf['is_night']  = (ddf['hour'] >= 22) | (ddf['hour'] <= 5)

night_by_era = (
    ddf.groupby('covid_era')
       .agg({'is_night': 'sum', 'unique_key': 'count'})
       .compute()
       .rename(columns={'is_night': 'night_crimes', 'unique_key': 'total'})
       .reset_index()
)
night_by_era['night_rate_pct'] = (night_by_era['night_crimes'] / night_by_era['total'] * 100).round(2)

pre_night     = float(night_by_era.loc[night_by_era['covid_era']=='PRE',    'night_rate_pct'].values[0])
durante_night = float(night_by_era.loc[night_by_era['covid_era']=='DURANTE','night_rate_pct'].values[0])
post_night    = float(night_by_era.loc[night_by_era['covid_era']=='POST',   'night_rate_pct'].values[0])
kpi7          = round(durante_night - pre_night, 2)

print('KPI 7 | Cambio en Índice de Criminalidad Nocturna por Era')
for _, row in night_by_era.sort_values('covid_era').iterrows():
    print(f"  {row['covid_era']:<8}: {row['night_rate_pct']:.2f}%  ({int(row['night_crimes']):,} de {int(row['total']):,})")
print(f"  ΔDURANTE-PRE : {kpi7:+.2f} pp")

kpis.append({'kpi_id': 7, 'kpi_name': 'Cambio en Índice de Criminalidad Nocturna',
             'value_numeric': kpi7,
             'value_text': f'PRE:{pre_night:.2f}% → DURANTE:{durante_night:.2f}% → POST:{post_night:.2f}%',
             'unit': 'pp', 'framework': 'Dask'})

KPI 7 | Cambio en Índice de Criminalidad Nocturna por Era
  DURANTE : 28.01%  (185,283 de 661,583)
  POST    : 27.99%  (171,178 de 611,521)
  PRE     : 24.42%  (195,300 de 799,839)
  ΔDURANTE-PRE : +3.59 pp


---
## KPI 8 — Variación de Crímenes Violentos (FBI Part I) por Era *(Spark)*

**Definición:** Cambio en la proporción de crímenes violentos (FBI Part I) entre PRE y DURANTE COVID.

**Fórmula:** `Δ = violent_rate_DURANTE - violent_rate_PRE` donde `violent_rate = SUM(fbi_code ∈ Part_I) / total × 100`

**Interpretación:** El COVID afectó asimétricamente los crímenes violentos vs no-violentos. Los crímenes de oportunidad (robo en comercios, carterismo) cayeron con el cierre de espacios públicos. Los crímenes domésticos y la violencia interpersonal en residencias (clasificados como Part I cuando incluyen armas) pudieron mantenerse o aumentar. Un incremento en violent_rate durante DURANTE indica un desplazamiento hacia crímenes más graves.

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('ChicagoCrimes-KPIs-COVID')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)

# Expresión CASE WHEN para covid_era en Spark
era_expr = F.when(F.col('year') <= 2019, 'PRE') \
            .when(F.col('year') <= 2022, 'DURANTE') \
            .otherwise('POST')

sdf = (
    spark.read
         .option('header', 'true')
         .option('nullValue', '')
         .option('inferSchema', 'true')
         .csv(files)
    .withColumn('date',      F.to_timestamp('date'))
    .withColumn('covid_era', era_expr)
    .filter(F.col('year').between(2017, 2025))
)
sdf.createOrReplaceTempView('crimes')

violent_by_era = spark.sql("""
    SELECT
        covid_era,
        COUNT(*) AS total,
        SUM(CASE WHEN fbi_code IN ('01A','01B','02','03','04A','04B','05','06','07','09')
                 THEN 1 ELSE 0 END) AS violent,
        ROUND(
            SUM(CASE WHEN fbi_code IN ('01A','01B','02','03','04A','04B','05','06','07','09')
                     THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
        ) AS violent_rate_pct
    FROM crimes
    WHERE covid_era IS NOT NULL
    GROUP BY covid_era
    ORDER BY covid_era
""")

violent_pdf = violent_by_era.toPandas()
pre_vr     = float(violent_pdf.loc[violent_pdf['covid_era']=='PRE',    'violent_rate_pct'].values[0])
durante_vr = float(violent_pdf.loc[violent_pdf['covid_era']=='DURANTE','violent_rate_pct'].values[0])
post_vr    = float(violent_pdf.loc[violent_pdf['covid_era']=='POST',   'violent_rate_pct'].values[0])
kpi8       = round(durante_vr - pre_vr, 2)

print('KPI 8 | Variación de Crímenes Violentos (FBI Part I) por Era')
print(violent_pdf[['covid_era','total','violent','violent_rate_pct']].to_string(index=False))
print(f'\n  ΔDURANTE-PRE : {kpi8:+.2f} pp')

kpis.append({'kpi_id': 8, 'kpi_name': 'Variación Crímenes Violentos FBI Part I por COVID',
             'value_numeric': kpi8,
             'value_text': f'PRE:{pre_vr:.2f}% → DURANTE:{durante_vr:.2f}% → POST:{post_vr:.2f}%',
             'unit': 'pp', 'framework': 'Spark'})

KPI 8 | Variación de Crímenes Violentos (FBI Part I) por Era
covid_era  total  violent violent_rate_pct
  DURANTE 661583   117144            17.71
     POST 611521   122668            20.06
      PRE 799839   121517            15.19

  ΔDURANTE-PRE : +2.52 pp


---
## KPI 9 — Diversidad de Crímenes por Era: Índice de Shannon *(Spark)*

**Definición:** Entropía de Shannon calculada sobre la distribución de tipos de crimen en cada era COVID. Mide si el COVID concentró o diversificó el perfil delictivo.

**Fórmula:** `H_era = -Σ(p_i × log₂(p_i))` donde `p_i = proporción del tipo i en el total de la era`

**Interpretación:** Una caída en H durante DURANTE indica concentración del crimen — menos tipos de delito dominan el total, probablemente porque los crímenes de oportunidad (robo en tiendas, carterismo) cayeron a cero mientras los domésticos y violentos se mantuvieron. Una recuperación en POST hacia los niveles PRE confirmaría la normalización del perfil delictivo.

In [11]:
proportions_era = spark.sql("""
    SELECT covid_era, primary_type,
        COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY covid_era) AS proportion
    FROM crimes
    WHERE covid_era IS NOT NULL AND primary_type IS NOT NULL
    GROUP BY covid_era, primary_type
""")

shannon_era = (
    proportions_era
    .withColumn('contrib', -F.col('proportion') * F.log2(F.col('proportion')))
    .groupBy('covid_era')
    .agg(F.round(F.sum('contrib'), 4).alias('shannon_index'))
    .orderBy('covid_era')
)

shannon_pdf = shannon_era.toPandas()
pre_h     = float(shannon_pdf.loc[shannon_pdf['covid_era']=='PRE',    'shannon_index'].values[0])
durante_h = float(shannon_pdf.loc[shannon_pdf['covid_era']=='DURANTE','shannon_index'].values[0])
post_h    = float(shannon_pdf.loc[shannon_pdf['covid_era']=='POST',   'shannon_index'].values[0])
kpi9      = round(durante_h - pre_h, 4)

print('KPI 9 | Diversidad de Crímenes por Era (Índice de Shannon)')
print(shannon_pdf.to_string(index=False))
print(f'\n  ΔDURANTE-PRE : {kpi9:+.4f} bits')
interp = 'CONCENTRACIÓN (menos diverso)' if kpi9 < 0 else 'DIVERSIFICACIÓN (más diverso)'
print(f'  Interpretación: {interp}')

kpis.append({'kpi_id': 9, 'kpi_name': 'Cambio en Diversidad de Crímenes (Shannon) por COVID',
             'value_numeric': kpi9,
             'value_text': f'PRE:{pre_h:.4f} → DURANTE:{durante_h:.4f} → POST:{post_h:.4f} bits',
             'unit': 'bits', 'framework': 'Spark'})

KPI 9 | Diversidad de Crímenes por Era (Índice de Shannon)
covid_era  shannon_index
  DURANTE         3.4674
     POST         3.4228
      PRE         3.4469

  ΔDURANTE-PRE : +0.0205 bits
  Interpretación: DIVERSIFICACIÓN (más diverso)


---
## KPI 10 — Variación en la Calidad del Registro CPD Durante el COVID (MTTR) *(Spark)*

**Definición:** Promedio de días entre la fecha del incidente (`date`) y la última actualización del registro (`updated_on`), calculado por era COVID. Mide si el COVID degradó la calidad de los registros policiales.

**Fórmula:** `MTTR_era = AVG(DATEDIFF(updated_on, date))` por era

**Interpretación:** Un MTTR mayor en la era DURANTE indica sobrecarga administrativa del CPD — los oficiales tardaron más en completar/actualizar los informes de incidentes por los protocolos COVID, reducción de personal de oficina, o transición a trabajo remoto parcial. Un MTTR recuperado en POST confirmaría que fue un efecto temporal de la pandemia.

In [12]:
mttr_by_era = spark.sql("""
    SELECT
        covid_era,
        ROUND(AVG(DATEDIFF(
            TO_DATE(CAST(updated_on AS STRING)),
            TO_DATE(CAST(date AS STRING))
        )), 1) AS avg_days,
        MIN(DATEDIFF(
            TO_DATE(CAST(updated_on AS STRING)),
            TO_DATE(CAST(date AS STRING))
        )) AS min_days
    FROM crimes
    WHERE date IS NOT NULL AND updated_on IS NOT NULL AND covid_era IS NOT NULL
    GROUP BY covid_era
    ORDER BY covid_era
""")

mttr_pdf  = mttr_by_era.toPandas()
pre_mttr     = float(mttr_pdf.loc[mttr_pdf['covid_era']=='PRE',    'avg_days'].values[0])
durante_mttr = float(mttr_pdf.loc[mttr_pdf['covid_era']=='DURANTE','avg_days'].values[0])
post_mttr    = float(mttr_pdf.loc[mttr_pdf['covid_era']=='POST',   'avg_days'].values[0])
kpi10        = round(durante_mttr - pre_mttr, 1)

print('KPI 10 | Variación en Calidad del Registro CPD (MTTR) por Era')
print(mttr_pdf[['covid_era','avg_days','min_days']].to_string(index=False))
print(f'\n  ΔDURANTE-PRE : {kpi10:+.1f} días')

kpis.append({'kpi_id': 10, 'kpi_name': 'Variación MTTR Registro CPD por COVID',
             'value_numeric': kpi10,
             'value_text': f'PRE:{pre_mttr:.1f}d → DURANTE:{durante_mttr:.1f}d → POST:{post_mttr:.1f}d',
             'unit': 'días', 'framework': 'Spark'})

KPI 10 | Variación en Calidad del Registro CPD (MTTR) por Era
covid_era  avg_days  min_days
  DURANTE      87.9         4
     POST     112.2        -2
      PRE     125.9         6

  ΔDURANTE-PRE : -38.0 días


In [13]:
# ── Consolidar y subir kpis_summary a BigQuery ───────────────────────────────
kpis_df = pd.DataFrame(kpis)

print('=' * 75)
print('RESUMEN EJECUTIVO — Chicago Crimes KPIs COVID-19 (2017–2025)')
print('=' * 75)
for _, row in kpis_df.iterrows():
    print(f"KPI {int(row['kpi_id']):02d} [{row['framework']:<5}] {row['kpi_name']:<45} → {row['value_text']}")
print('=' * 75)

to_bigquery(kpis_df, 'kpis_summary')

RESUMEN EJECUTIVO — Chicago Crimes KPIs COVID-19 (2017–2025)
KPI 01 [Dask ] Variación Tasa de Arresto PRE→DURANTE         → PRE:20.36% → DURANTE:13.42% → POST:13.32%
KPI 02 [Dask ] Caída de Criminalidad en Confinamiento 2020   → -18.75% (2019→2020)
KPI 03 [Modin] Aumento de Violencia Doméstica DURANTE COVID  → PRE:19.20% → DURANTE:21.02% → POST:18.39%
KPI 04 [Modin] Tipo con Mayor Cambio de Resolución por COVID → PUBLIC PEACE VIOLATION: -27.88 pp (PRE→DURANTE)
KPI 05 [Modin] Distrito con Mayor Caída de Crimen en COVID   → Distrito 1: -33.89% (DURANTE vs PRE anualizado)
KPI 06 [Dask ] Recuperación Post-COVID vs Nivel PRE          → -23.54% (POST vs PRE anualizado)
KPI 07 [Dask ] Cambio en Índice de Criminalidad Nocturna     → PRE:24.42% → DURANTE:28.01% → POST:27.99%
KPI 08 [Spark] Variación Crímenes Violentos FBI Part I por COVID → PRE:15.19% → DURANTE:17.71% → POST:20.06%
KPI 09 [Spark] Cambio en Diversidad de Crímenes (Shannon) por COVID → PRE:3.4469 → DURANTE:3.4674 → POST:3.4228 bi

  → BigQuery: my-first-project-492901.chicago_crimes_results.kpis_summary  (10 filas)


In [14]:
spark.stop()
ray.shutdown()
print('Entorno cerrado. Todos los KPIs guardados en BigQuery.')

Entorno cerrado. Todos los KPIs guardados en BigQuery.
